# 09 — Préparer un dataset

Nettoie les lignes invalides et effectue une séparation **temporelle** entraînement/validation. Ne transforme pas des mesures de volant en actions de pilotage sans vérifier leur signification.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'OUARTI'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier OUARTI absent : extraire OUARTI a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


In [ ]:
import json,pandas as pd,numpy as np
files=sorted((PROJECT/'datasets').glob('session_*/telemetry.jsonl'))
if not files:raise FileNotFoundError('Collecte une session dabord.')
df=pd.DataFrame([json.loads(l) for l in files[-1].read_text(encoding='utf8').splitlines() if l.strip()])
needed=['speed','yaw','angular_velocity_y','accelX','accelY','steerAngle','accStatus','brakeStatus']
existing=[n for n in needed if n in df]
if len(existing)<3:raise ValueError('Champs telemetrie insuffisants : '+str(existing))
for c in existing:df[c]=pd.to_numeric(df[c],errors='coerce')
df=df.replace([np.inf,-np.inf],np.nan).dropna(subset=existing).reset_index(drop=True)
if len(df)<20:raise ValueError('Moins de 20 echantillons valides, enregistrer davantage.')
pivot=int(len(df)*.8)
train=df.iloc[:pivot]; valid=df.iloc[pivot:]
train.to_csv(PROJECT/'datasets'/'train_telemetry.csv',index=False)
valid.to_csv(PROJECT/'datasets'/'valid_telemetry.csv',index=False)
print('Train :',len(train),'Validation :',len(valid))
